# 00 — Setup, download, AnnData and SpatialData

Interactive counterpart of HPC steps 00–02. The heavy operations still use the tested production scripts, but each stage is a separate cell so it can be inspected/re-run independently.

In [1]:
from pathlib import Path
import sys, os

# Notebook lives in PROJECT_ROOT/jupyter/.
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "jupyter" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT))
print("PROJECT_ROOT:", PROJECT_ROOT)
print("Python:", sys.executable)
assert (PROJECT_ROOT / "scripts").exists(), "Run this notebook from PROJECT_ROOT/jupyter or PROJECT_ROOT."

PROJECT_ROOT: /research/rgs01/home/clusterHome/jqu/activities/learning/BioHackathon/BioHackathon-2026/Dataset_02_CosMx_revised/Dataset_02_CosMx_revised
Python: /home/jqu/.conda/envs/spatialdata/bin/python


## Optional: download the five GEO CosMx files

In [ ]:
%run ../scripts/00_download_cosmx.py

## Build raw AnnData

In [ ]:
%run ../scripts/01_build_anndata.py

## Inspect raw AnnData

In [2]:
import scanpy as sc
RAW_H5AD=PROJECT_ROOT/"data/processed/GSM9046088_CosMx_raw.h5ad"
a=sc.read_h5ad(RAW_H5AD)
print(a)
display(a.obs.head())
display(a.var.head())

AnnData object with n_obs × n_vars = 43565 × 1010
    obs: 'fov', 'cell_ID', 'Area', 'AspectRatio', 'CenterX_local_px', 'CenterY_local_px', 'CenterX_global_px', 'CenterY_global_px', 'Width', 'Height', 'Mean.PanCK', 'Max.PanCK', 'Mean.CD68', 'Max.CD68', 'Mean.B2M.MembraneStain', 'Max.B2M.MembraneStain', 'Mean.CD45', 'Max.CD45', 'Mean.DAPI', 'Max.DAPI', 'n_genes_by_counts', 'total_counts'
    var: 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'
    uns: 'GEO_accession', 'fov_positions', 'platform', 'sample', 'spatial_coordinate_columns'
    obsm: 'spatial'
    layers: 'counts'


,fov,cell_ID,Area,AspectRatio,CenterX_local_px,CenterY_local_px,CenterX_global_px,CenterY_global_px,Width,Height,...,Mean.CD68,Max.CD68,Mean.B2M.MembraneStain,Max.B2M.MembraneStain,Mean.CD45,Max.CD45,Mean.DAPI,Max.DAPI,n_genes_by_counts,total_counts
unique_cell_id,,,,,,,,,,,,,,,,,,,,,
fov_1_cell_1,1,1,7538,0.95,2515,4056,-495476.6667,11339.33333,99,104,...,0,105,108,458,2,128,1,87,39,106.0
fov_1_cell_2,1,2,4969,1.11,3831,3925,-494160.6667,11208.33333,83,75,...,0,93,53,787,3,530,34,306,76,96.0
fov_1_cell_3,1,3,4045,0.75,2368,3901,-495623.6667,11184.33333,65,87,...,5,829,344,1389,18,1284,105,397,87,156.0
fov_1_cell_4,1,4,7853,0.86,2921,3695,-495070.6667,10978.33333,96,111,...,18,245,172,843,29,589,136,550,192,308.0
fov_1_cell_5,1,5,7992,0.81,3907,3665,-494084.6667,10948.33333,90,111,...,6,310,96,1015,16,909,0,34,17,18.0


,n_cells_by_counts,mean_counts,pct_dropout_by_counts,total_counts
gene,,,,
RAMP2,3366,0.104028,92.273614,4532.0
CD83,4873,0.163549,88.814415,7125.0
RYK,3899,0.110662,91.050155,4821.0
CD5L,2814,0.082337,93.540686,3587.0
NLRP2,2719,0.079582,93.758751,3467.0


## Build SpatialData

In [ ]:
%run ../scripts/02_build_sdata.py

## Inspect SpatialData

In [4]:
import spatialdata as sd

SDATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "spatial"
    / "GSM9046088_CosMx.zarr"
)

S = sd.read_zarr(SDATA_PATH)

print("read_zarr succeeded")
print(type(S))

read_zarr succeeded
<class 'spatialdata._core.spatialdata.SpatialData'>


In [5]:
print("Points:", list(S.points.keys()))
print("Shapes:", list(S.shapes.keys()))
print("Tables:", list(S.tables.keys()))
print("Images:", list(S.images.keys()))
print("Labels:", list(S.labels.keys()))
print("Coordinate systems:", S.coordinate_systems)

Points: ['transcripts']
Shapes: ['cell_boundaries']
Tables: ['table']
Images: []
Labels: []
Coordinate systems: ['global']


In [6]:
shapes = S.shapes["cell_boundaries"]

print(type(shapes))
print(shapes.shape)
print(shapes.head())

<class 'geopandas.geodataframe.GeoDataFrame'>
(43565, 1)
                                                       geometry
instance_id                                                    
fov_1_cell_1  POLYGON ((-495476.667 11390.333, -495469.667 1...
fov_1_cell_2  POLYGON ((-494179.667 11245.333, -494174.667 1...
fov_1_cell_3  POLYGON ((-495625.667 11227.333, -495620.667 1...
fov_1_cell_4  POLYGON ((-495069.667 11033.333, -495064.667 1...
fov_1_cell_5  POLYGON ((-494089.667 11003.333, -494084.667 1...


In [7]:
tx = S.points["transcripts"]

print(type(tx))
print(tx.columns)
print(tx.dtypes)

<class 'dask.dataframe.dask_expr._collection.DataFrame'>
Index(['x', 'y', 'target', 'cell_ID', 'x_local_px', 'fov', 'y_local_px',
       'CellComp', 'Unnamed: 0'],
      dtype='object')
x                     float64
y                     float64
target               category
cell_ID               float64
x_local_px            float64
fov                   float64
y_local_px            float64
CellComp      string[pyarrow]
Unnamed: 0            float64
dtype: object


In [9]:
import spatialdata
import dask
import pandas
import zarr

print("spatialdata :", spatialdata.__version__)
print("dask        :", dask.__version__)
print("pandas      :", pandas.__version__)
print("zarr        :", zarr.__version__)

spatialdata : 0.7.2
dask        : 2026.1.1
pandas      : 2.3.3
zarr        : 3.1.6


In [10]:
import spatialdata
import spatialdata_plot
import scanpy
import anndata
import dask
import pandas
import zarr

print("Package versions")
print("----------------")
print("spatialdata     :", spatialdata.__version__)
print("spatialdata-plot:", spatialdata_plot.__version__)
print("scanpy          :", scanpy.__version__)
print("anndata         :", anndata.__version__)
print("dask            :", dask.__version__)
print("pandas          :", pandas.__version__)
print("zarr            :", zarr.__version__)

Package versions
----------------
spatialdata     : 0.7.2
spatialdata-plot: 0.3.3
scanpy          : 1.11.5
anndata         : 0.12.11
dask            : 2026.1.1
pandas          : 2.3.3
zarr            : 3.1.6


/lsf_tmp/323066199.tmpdir/ipykernel_1501017/2809436416.py:13: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print("scanpy          :", scanpy.__version__)
/lsf_tmp/323066199.tmpdir/ipykernel_1501017/2809436416.py:14: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print("anndata         :", anndata.__version__)


In [11]:
import spatialdata as sd


SDATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "spatial"
    / "GSM9046088_CosMx.zarr"
)

S = sd.read_zarr(SDATA_PATH)

print("SpatialData loaded successfully")
print(f"Path: {SDATA_PATH}")
print()

print("Elements:")
print("  Images :", list(S.images.keys()))
print("  Labels :", list(S.labels.keys()))
print("  Points :", list(S.points.keys()))
print("  Shapes :", list(S.shapes.keys()))
print("  Tables :", list(S.tables.keys()))
print()

print(
    "Coordinate systems:",
    S.coordinate_systems,
)

for name, table in S.tables.items():
    print(
        f"Table {name!r}: "
        f"{table.n_obs:,} cells × "
        f"{table.n_vars:,} features"
    )

for name, shapes in S.shapes.items():
    print(
        f"Shapes {name!r}: "
        f"{len(shapes):,} polygons"
    )

for name, points in S.points.items():
    print(
        f"Points {name!r}: "
        f"columns={list(points.columns)}"
    )

SpatialData loaded successfully
Path: /research/rgs01/home/clusterHome/jqu/activities/learning/BioHackathon/BioHackathon-2026/Dataset_02_CosMx_revised/Dataset_02_CosMx_revised/data/spatial/GSM9046088_CosMx.zarr

Elements:
  Images : []
  Labels : []
  Points : ['transcripts']
  Shapes : ['cell_boundaries']
  Tables : ['table']

Coordinate systems: ['global']
Table 'table': 43,565 cells × 1,010 features
Shapes 'cell_boundaries': 43,565 polygons
Points 'transcripts': columns=['x', 'y', 'target', 'cell_ID', 'x_local_px', 'fov', 'y_local_px', 'CellComp', 'Unnamed: 0']
